In [1]:
import sys
sys.path.append("..")   # taaki src folder import ho sake

import ee, geemap
from src.config import GEE_PROJECT, AOI_BOX

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_BOX)
print("Ready")

Ready


In [2]:
def s1_collection(start, end, orbit):
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("orbitProperties_pass", orbit))
            .select("VV"))

def list_dates(start, end, orbit):
    col = s1_collection(start, end, orbit)
    dates = col.aggregate_array("system:time_start").map(
        lambda t: ee.Date(t).format("YYYY-MM-dd"))
    return sorted(set(dates.getInfo()))

for orbit in ["ASCENDING", "DESCENDING"]:
    print(orbit, list_dates("2022-08-15", "2022-10-15", orbit))

ASCENDING ['2022-08-18', '2022-08-30', '2022-09-11', '2022-09-23', '2022-10-05']
DESCENDING ['2022-08-16', '2022-08-21', '2022-08-28', '2022-09-02', '2022-09-14', '2022-09-21', '2022-09-26', '2022-10-03', '2022-10-08']


In [3]:
def flood_map(before, after, orbit, diff_thr=-3, water_thr=-15):
    b = s1_collection(*before, orbit).mean().focal_median(30, "circle", "meters")
    a = s1_collection(*after, orbit).mean().focal_median(30, "circle", "meters")

    diff = a.subtract(b)
    # naya paani = bahut dark ho gaya (diff) AND ab bhi absolute dark hai
    flood = diff.lt(diff_thr).And(a.lt(water_thr))

    # permanent paani (nadi, samudra) hatao
    perm = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50)
    flood = flood.updateMask(perm.Not())

    # steep slope hatao
    slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
    flood = flood.updateMask(slope.lt(5)).selfMask()
    return b.clip(aoi), a.clip(aoi), flood.clip(aoi)

# NOTE: ye dates sirf example hain, Step 2 ke output se replace karo
BEFORE = ("2022-08-20", "2022-09-05")
AFTER  = ("2022-09-14", "2022-09-28")
ORBIT  = "DESCENDING"

before_img, after_img, flood = flood_map(BEFORE, AFTER, ORBIT)

area = (flood.multiply(ee.Image.pixelArea())
        .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
        .get("VV"))
print("Flooded area (sq km):", ee.Number(area).divide(1e6).getInfo())

Flooded area (sq km): 17.2452300275338


In [4]:
m = geemap.Map(center=[20.55, 86.6], zoom=9)
m.addLayer(before_img, {"min": -25, "max": 0}, "Before")
m.addLayer(after_img, {"min": -25, "max": 0}, "After")
m.addLayer(flood, {"palette": ["blue"]}, "Flood")
m

Map(center=[20.55, 86.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [5]:
# Baseline 1: monsoon se pehle, dry season (kam geele khet)
BASELINES = {
    "dry (Mar-Apr)": ("2022-03-01", "2022-04-15"),
    "early monsoon (Aug 4-16)": ("2022-08-04", "2022-08-17"),
}

AFTER_WINDOWS = [
    ("2022-08-16", "2022-08-28"),
    ("2022-08-28", "2022-09-09"),
    ("2022-09-09", "2022-09-21"),
    ("2022-09-21", "2022-10-03"),
    ("2022-10-03", "2022-10-15"),
]

def flood_area(before, after, orbit="DESCENDING"):
    _, _, fl = flood_map(before, after, orbit)
    a = (fl.multiply(ee.Image.pixelArea())
         .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
         .get("VV"))
    return round(ee.Number(a).divide(1e6).getInfo(), 1)

for bname, b in BASELINES.items():
    print("Baseline:", bname)
    for w in AFTER_WINDOWS:
        print("   ", w, "->", flood_area(b, w), "sq km")

Baseline: dry (Mar-Apr)
    ('2022-08-16', '2022-08-28') -> 407.3 sq km
    ('2022-08-28', '2022-09-09') -> 191.0 sq km
    ('2022-09-09', '2022-09-21') -> 18.6 sq km
    ('2022-09-21', '2022-10-03') -> 48.7 sq km
    ('2022-10-03', '2022-10-15') -> 38.7 sq km
Baseline: early monsoon (Aug 4-16)
    ('2022-08-16', '2022-08-28') -> 287.6 sq km
    ('2022-08-28', '2022-09-09') -> 112.4 sq km
    ('2022-09-09', '2022-09-21') -> 7.2 sq km
    ('2022-09-21', '2022-10-03') -> 18.0 sq km
    ('2022-10-03', '2022-10-15') -> 15.2 sq km


In [6]:
BEFORE = ("2022-08-04", "2022-08-17")
AFTER  = ("2022-08-18", "2022-08-24")

for orbit in ["ASCENDING", "DESCENDING"]:
    try:
        b, a, fl = flood_map(BEFORE, AFTER, orbit)
        area = (fl.multiply(ee.Image.pixelArea())
                .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
                .get("VV"))
        print(orbit, "->", round(ee.Number(area).divide(1e6).getInfo(), 1), "sq km")
    except Exception as e:
        print(orbit, "error:", e)

ASCENDING -> 341.2 sq km
DESCENDING -> 29.9 sq km


In [7]:
ORBIT = "DESCENDING"   # ya "ASCENDING", jo zyada area de
b, a, fl = flood_map(BEFORE, AFTER, ORBIT)

m = geemap.Map(center=[20.55, 86.6], zoom=9)
m.addLayer(b, {"min": -25, "max": 0}, "Before")
m.addLayer(a, {"min": -25, "max": 0}, "After")
m.addLayer(fl, {"palette": ["blue"]}, "Flood")
m

Map(center=[20.55, 86.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [8]:
BEFORE = ("2022-08-04", "2022-08-17")
AFTER  = ("2022-08-18", "2022-08-24")
b, a, fl = flood_map(BEFORE, AFTER, "DESCENDING")

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(a, {"min": -25, "max": 0}, "After")
m.addLayer(fl, {"palette": ["0000FF"]}, "Flood")
m

Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
BEFORE = ("2022-08-04", "2022-08-17")
AFTER  = ("2022-08-18", "2022-08-24")

_, a_d, fl_d = flood_map(BEFORE, AFTER, "DESCENDING")
_, a_a, fl_a = flood_map(BEFORE, AFTER, "ASCENDING")

# dono ko jodo
flood_all = ee.ImageCollection([fl_d.unmask(0), fl_a.unmask(0)]).max().selfMask()

area = (flood_all.multiply(ee.Image.pixelArea())
        .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
        .getInfo())
print(area)   # dictionary aayegi, uski value /1e6 karke sq km

m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(a_d, {"min": -25, "max": 0}, "After DESC")
m.addLayer(a_a, {"min": -25, "max": 0}, "After ASC")
m.addLayer(flood_all, {"palette": ["0000FF"]}, "Flood (merged)")
m.addLayer(ee.Image().paint(aoi, 0, 2), {}, "AOI boundary")
m

{'VV': 351009629.62088937}


Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
# 8+ connected pixels (~0.7 hectare) se chhote patches hatao
conn = flood_all.connectedPixelCount(maxSize=100, eightConnected=True)
flood_clean = flood_all.updateMask(conn.gte(8))

def area_km2(img):
    d = (img.multiply(ee.Image.pixelArea())
         .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10).getInfo())
    return round(list(d.values())[0] / 1e6, 1)

print("Merged flood :", area_km2(flood_all), "sq km")
print("Cleaned flood:", area_km2(flood_clean), "sq km")

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(a_d, {"min": -25, "max": 0}, "After DESC")
m.addLayer(flood_clean, {"palette": ["0000FF"]}, "Flood (clean)")
m

Merged flood : 351.0 sq km
Cleaned flood: 288.9 sq km


Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [11]:
task = ee.batch.Export.image.toDrive(
    image=flood_clean.unmask(0).toByte(),
    description="flood_mask_aug2022",
    folder="jalrakshak",
    fileNamePrefix="flood_mask_aug2022",
    region=aoi,
    scale=30,
    maxPixels=1e10,
)
task.start()
print("Export started. Check Drive folder 'jalrakshak' in 2-5 minutes.")

Export started. Check Drive folder 'jalrakshak' in 2-5 minutes.


In [12]:
import sys
sys.path.append("..")

# 1. Shelters
from src.shelters import find_shelters
try:
    s = find_shelters(20.50, 86.60)
    print("Shelters mile:", len(s))
    for x in s[:3]:
        print(x)
except Exception as e:
    print("SHELTER ERROR:", repr(e))

# 2. Telegram
import os
from dotenv import load_dotenv
load_dotenv("../.env")
print("Token set:", bool(os.getenv("TELEGRAM_BOT_TOKEN")), "| Chat id set:", bool(os.getenv("TELEGRAM_CHAT_ID")))
from src.alerts import send_telegram
try:
    print(send_telegram("JalRakshak test message"))
except Exception as e:
    print("TELEGRAM ERROR:", repr(e))

Shelters mile: 5
{'name': 'Sri Krishna Health Care and Reproductive Research Centre', 'type': 'hospital', 'lat': 20.4961016, 'lon': 86.6202067, 'dist_km': 2.15}
{'name': 'Government Hospital, Ayeba', 'type': 'hospital', 'lat': 20.5118115, 'lon': 86.5202335, 'dist_km': 8.41}
{'name': 'Mahakalpara Hospital', 'type': 'hospital', 'lat': 20.4224275, 'lon': 86.5790084, 'dist_km': 8.9}
Token set: True | Chat id set: True
{'ok': True, 'result': {'message_id': 9, 'from': {'id': 9221415665, 'is_bot': True, 'first_name': 'JalRakshak Bot', 'username': 'JalRakshakDanger_bot'}, 'chat': {'id': 8947775720, 'first_name': 'Chandan', 'last_name': 'Pradhan', 'type': 'private'}, 'date': 1794469038, 'text': 'JalRakshak test message'}}


In [13]:
import sys
print("start")
sys.path.append("..")

import requests
print("requests OK")

url = "https://overpass-api.de/api/interpreter"
q = '[out:json][timeout:25];nwr["amenity"="school"](around:10000,20.5,86.6);out center 5;'
try:
    r = requests.post(
        url, data={"data": q},
        headers={"User-Agent": "JalRakshak-student-project/1.0 (final-year project, Odisha)"},
        timeout=40,
    )
    print("status:", r.status_code)
    print("elements:", len(r.json().get("elements", [])))
except Exception as e:
    print("ERROR:", repr(e))

import os
print(".env exists:", os.path.exists("../.env"))
print(open("../.env").read() if os.path.exists("../.env") else "no .env file")

start
requests OK
status: 502
ERROR: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
.env exists: True
TELEGRAM_BOT_TOKEN=9221415665:AAGOYFIaJOFMc8lS47HgD7Lenv1pb6UTKD0
TELEGRAM_CHAT_ID=8947775720



In [14]:
import os, sys, requests
print("START")

# 1. .env ka haal (token chhupa ke)
path = os.path.abspath("../.env")
print(".env path:", path)
txt = open(path).read() if os.path.exists(path) else ""
print(".env size (characters):", len(txt))
for line in txt.splitlines():
    k = line.split("=")[0]
    print("  line:", k, "= (value set)" if "=" in line and line.split("=", 1)[1].strip() else "= (KHAALI)")

# 2. Overpass test, 3 servers
H = {"User-Agent": "JalRakshak-student-project/1.0 (final-year project, Odisha)"}
q = '[out:json][timeout:25];nwr["amenity"="school"](around:10000,20.5,86.6);out center 5;'
for u in ["https://overpass-api.de/api/interpreter",
          "https://overpass.kumi.systems/api/interpreter",
          "https://overpass.private.coffee/api/interpreter"]:
    try:
        r = requests.post(u, data={"data": q}, headers=H, timeout=40)
        print("OVERPASS", u.split("/")[2], "->", r.status_code)
    except Exception as e:
        print("OVERPASS", u.split("/")[2], "-> ERROR", type(e).__name__)

print("END")

START
.env path: c:\Users\pradh\OneDrive\Desktop\jalrakshak\.env
.env size (characters): 94
  line: TELEGRAM_BOT_TOKEN = (value set)
  line: TELEGRAM_CHAT_ID = (value set)
OVERPASS overpass-api.de -> 502
OVERPASS overpass.kumi.systems -> ERROR ReadTimeout
OVERPASS overpass.private.coffee -> ERROR ReadTimeout
END


In [15]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)
from src.alerts import send_telegram
print(send_telegram("JalRakshak test message"))

{'ok': True, 'result': {'message_id': 10, 'from': {'id': 9221415665, 'is_bot': True, 'first_name': 'JalRakshak Bot', 'username': 'JalRakshakDanger_bot'}, 'chat': {'id': 8947775720, 'first_name': 'Chandan', 'last_name': 'Pradhan', 'type': 'private'}, 'date': 1794469139, 'text': 'JalRakshak test message'}}


In [16]:
import os
p = os.path.abspath("../.env")
txt = open(p, encoding="utf-8-sig").read() if os.path.exists(p) else ""
print("path:", p)
print("size:", len(txt))
for line in txt.splitlines():
    k, _, v = line.partition("=")
    print(repr(k), "-> value length:", len(v.strip()))

path: c:\Users\pradh\OneDrive\Desktop\jalrakshak\.env
size: 94
'TELEGRAM_BOT_TOKEN' -> value length: 46
'TELEGRAM_CHAT_ID' -> value length: 10


In [17]:
token = input("Bot token paste karo: ").strip()

with open("../.env", "w", encoding="utf-8") as f:
    f.write(f"TELEGRAM_BOT_TOKEN={token}\n")
    f.write("TELEGRAM_CHAT_ID=8947775720\n")

print("Saved. Size:", len(open("../.env").read()))

Saved. Size: 94


In [18]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)
import sys; sys.path.append("..")
from src.alerts import send_telegram
print(send_telegram("JalRakshak test message"))

{'ok': True, 'result': {'message_id': 11, 'from': {'id': 9221415665, 'is_bot': True, 'first_name': 'JalRakshak Bot', 'username': 'JalRakshakDanger_bot'}, 'chat': {'id': 8947775720, 'first_name': 'Chandan', 'last_name': 'Pradhan', 'type': 'private'}, 'date': 1794469596, 'text': 'JalRakshak test message'}}


In [19]:
token = input("Bot token paste karo: ").strip()
with open("../.env", "w", encoding="utf-8") as f:
    f.write(f"TELEGRAM_BOT_TOKEN={token}\n")
    f.write("TELEGRAM_CHAT_ID=8947775720\n")
print("Saved. Size:", len(open("../.env").read()))

Saved. Size: 48


In [20]:
   import sys; sys.path.append("..")
   from src.shelters import find_shelters
   s = find_shelters(20.80, 86.40)
   print(len(s)); print(s[:3])

5
[{'name': 'Bhelanga Homeopthy Hospital', 'type': 'hospital', 'lat': 20.8136504, 'lon': 86.3786012, 'dist_km': 2.69}, {'name': 'Government Hospital, Barpada', 'type': 'hospital', 'lat': 20.7481673, 'lon': 86.437199, 'dist_km': 6.94}, {'name': 'Niramaya Kendra', 'type': 'hospital', 'lat': 20.7250923, 'lon': 86.4040109, 'dist_km': 8.34}]


In [21]:
import sys; sys.path.insert(0, "..")
import importlib, src.alerts as a
importlib.reload(a)
print(a.send_telegram("JalRakshak test 2"))

RuntimeError: .env mein TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID nahi mila

In [ ]:
import sys; sys.path.insert(0, "..")
import importlib, src.alerts as a
importlib.reload(a)

fake_shelters = [{"name": "Government Hospital, Barpada", "dist_km": 6.94}]
msg = a.build_message("High", 150.0, fake_shelters)
print(msg)
print(a.send_telegram(msg))

ବନ୍ୟା ସତର୍କତା (ପ୍ରୋଟୋଟାଇପ୍)
ବିପଦ ସ୍ତର: High
ଆଗାମୀ ୩ ଦିନର ବର୍ଷା: 150.0 mm
ନିକଟତମ ଆଶ୍ରୟସ୍ଥଳ: Government Hospital, Barpada (6.94 km)

FLOOD ALERT (prototype)
Risk: High
Forecast rain (3 days): 150.0 mm
Nearest shelter: Government Hospital, Barpada (6.94 km)
Official warnings: IMD / OSDMA.
{'ok': True, 'result': {'message_id': 7, 'from': {'id': 9221415665, 'is_bot': True, 'first_name': 'JalRakshak Bot', 'username': 'JalRakshakDanger_bot'}, 'chat': {'id': 8947775720, 'first_name': 'Chandan', 'last_name': 'Pradhan', 'type': 'private'}, 'date': 1794394468, 'text': 'ବନ୍ୟା ସତର୍କତା (ପ୍ରୋଟୋଟାଇପ୍)\nବିପଦ ସ୍ତର: High\nଆଗାମୀ ୩ ଦିନର ବର୍ଷା: 150.0 mm\nନିକଟତମ ଆଶ୍ରୟସ୍ଥଳ: Government Hospital, Barpada (6.94 km)\n\nFLOOD ALERT (prototype)\nRisk: High\nForecast rain (3 days): 150.0 mm\nNearest shelter: Government Hospital, Barpada (6.94 km)\nOfficial warnings: IMD / OSDMA.'}}


In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/features_2022_v2.csv")
print(df.population.describe().round(1))
print("Sum of cell means:", round(df.population.sum()))

count    4544.0
mean        6.5
std         6.6
min         0.1
25%         4.1
50%         5.9
75%         7.5
max       214.0
Name: population, dtype: float64
Sum of cell means: 29742


In [ ]:
m = geemap.Map()
m.centerObject(aoi, 10)
m.add_basemap("SATELLITE")
m.addLayer(flood_clean, {"palette": ["0000FF"]}, "Flood")
m

NameError: name 'geemap' is not defined

In [ ]:
import sys
sys.path.append("..")
import ee, geemap
from src.config import GEE_PROJECT, AOI_BOX

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_BOX)

def s1_collection(start, end, orbit):
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(aoi).filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("orbitProperties_pass", orbit))
            .select("VV"))

def flood_map(before, after, orbit, diff_thr=-3, water_thr=-15):
    b = s1_collection(*before, orbit).mean().focal_median(30, "circle", "meters")
    a = s1_collection(*after, orbit).mean().focal_median(30, "circle", "meters")
    flood = (a.subtract(b)).lt(diff_thr).And(a.lt(water_thr))
    perm = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50)
    flood = flood.updateMask(perm.Not())
    slope_ = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
    return flood.updateMask(slope_.lt(5)).selfMask().clip(aoi)

def event_flood(before, after):
    fd = flood_map(before, after, "DESCENDING")
    fa = flood_map(before, after, "ASCENDING")
    merged = ee.ImageCollection([fd.unmask(0), fa.unmask(0)]).max().selfMask()
    conn = merged.connectedPixelCount(maxSize=100, eightConnected=True)
    return merged.updateMask(conn.gte(8))

flood_clean = event_flood(("2022-08-04", "2022-08-17"), ("2022-08-18", "2022-08-24"))

m = geemap.Map()
m.centerObject(aoi, 10)
m.add_basemap("SATELLITE")
m.addLayer(flood_clean, {"palette": ["0000FF"]}, "Flood")
m

Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [22]:
import pandas as pd
r = pd.read_csv("../data/processed/risk_map_2022.csv")
print(r[r.risk_level == "High"].sort_values("pop_est", ascending=False)[["lat","lon","pop_est"]].head(5))

         lat     lon  pop_est
1804  20.745  86.455   1485.0
1662  20.725  86.435   1425.0
2448  20.885  86.545   1389.0
1732  20.725  86.445   1388.0
2657  20.875  86.575   1360.0
